# 1. Data loading and preprocessing

EEEM075 - AI and Sustainability coursework

Loads the 5 years of UCI Gas Turbine CO/NOx data (2011-2015), checks data quality, builds a chronological train/val/test split (no shuffling, to avoid leakage across time-ordered sensor readings), and saves the split artefacts for the next notebooks.

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
ARTIFACT_DIR = Path('../artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

YEARS = range(2011, 2016)

## Load and tag by year

Each yearly file is loaded separately and tagged with its `year` before concatenation, since the year is what split on later (not a random shuffle).

In [35]:
frames = []
for y in YEARS:
    df_year = pd.read_csv(DATA_DIR / f'gt_{y}.csv')
    df_year['year'] = y
    frames.append(df_year)

df = pd.concat(frames, ignore_index=True)
print('Combined shape:', df.shape)
df.head()

Combined shape: (36733, 12)


,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX,year
0,4.5878,1018.7,83.675,3.5758,23.979,1086.2,549.83,134.67,11.898,0.32663,81.952,2011
1,4.2932,1018.3,84.235,3.5709,23.951,1086.1,550.05,134.67,11.892,0.44784,82.377,2011
2,3.9045,1018.4,84.858,3.5828,23.990,1086.5,550.19,135.10,12.042,0.45144,83.776,2011
3,3.7436,1018.3,85.434,3.5808,23.911,1086.5,550.17,135.03,11.990,0.23107,82.505,2011
4,3.7516,1017.8,85.182,3.5781,23.917,1085.9,550.00,134.67,11.910,0.26747,82.028,2011


## Data overview and data dictionary

Structural overview (row count, dtypes, memory footprint) plus a data dictionary describing each variable, its unit, and its role in this project. Variable descriptions follow the UCI repository documentation for this dataset (Gas Turbine CO and NOx Emission Data Set).

All 11 measured variables are continuous floats; `year` is added by the loading step above as the split key. The 9 ambient/operational sensors are the model inputs - `NOX` is the prediction target. `CO`, the other measured emission, is neither a target nor an input in this project: a PEMS soft sensor must infer emissions from cheap process sensors, so using one pollutant reading to predict another would defeat the purpose and leak target-adjacent information.

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36733 entries, 0 to 36732
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      36733 non-null  float64
 1   AP      36733 non-null  float64
 2   AH      36733 non-null  float64
 3   AFDP    36733 non-null  float64
 4   GTEP    36733 non-null  float64
 5   TIT     36733 non-null  float64
 6   TAT     36733 non-null  float64
 7   TEY     36733 non-null  float64
 8   CDP     36733 non-null  float64
 9   CO      36733 non-null  float64
 10  NOX     36733 non-null  float64
 11  year    36733 non-null  int64  
dtypes: float64(11), int64(1)
memory usage: 3.4 MB


In [37]:
data_dictionary = pd.DataFrame([
    ('AT',   'Ambient temperature',            '°C',    'input feature'),
    ('AP',   'Ambient pressure',               'mbar',  'input feature'),
    ('AH',   'Ambient humidity',               '%',     'input feature'),
    ('AFDP', 'Air filter difference pressure', 'mbar',  'input feature'),
    ('GTEP', 'Gas turbine exhaust pressure',   'mbar',  'input feature'),
    ('TIT',  'Turbine inlet temperature',      '°C',    'input feature'),
    ('TAT',  'Turbine after temperature',      '°C',    'input feature'),
    ('TEY',  'Turbine energy yield',           'MWh',   'input feature'),
    ('CDP',  'Compressor discharge pressure',  'mbar',  'input feature'),
    ('CO',   'Carbon monoxide concentration',  'mg/m³', 'emission variable — excluded from inputs'),
    ('NOX',  'Nitrogen oxides concentration',  'mg/m³', 'prediction target'),
    ('year', 'Year of the recording (added at load time)', '-', 'split key only — never a model input'),
], columns=['Variable', 'Description', 'Unit', 'Role in this project'])

data_dictionary

,Variable,Description,Unit,Role in this project
0,AT,Ambient temperature,°C,input feature
1,AP,Ambient pressure,mbar,input feature
2,AH,Ambient humidity,%,input feature
3,AFDP,Air filter difference pressure,mbar,input feature
4,GTEP,Gas turbine exhaust pressure,mbar,input feature
5,TIT,Turbine inlet temperature,°C,input feature
6,TAT,Turbine after temperature,°C,input feature
7,TEY,Turbine energy yield,MWh,input feature
8,CDP,Compressor discharge pressure,mbar,input feature
9,CO,Carbon monoxide concentration,mg/m³,emission variable — excluded from inputs


## Data quality checks

Check three things before doing any statistical outlier analysis:
1. Missing values.
2. Physically impossible readings (values that violate the physical meaning of the sensor, regardless of how 'normal' they look statistically).
3. Statistical outliers (IQR), reported separately — these are *not* automatically removed, since a gas turbine legitimately operates across a wide load range and extreme-but-real operating points are exactly what the model needs to see.

In [38]:
print('Missing values per column:')
print(df.isnull().sum())

Missing values per column:
AT      0
AP      0
AH      0
AFDP    0
GTEP    0
TIT     0
TAT     0
TEY     0
CDP     0
CO      0
NOX     0
year    0
dtype: int64


In [39]:
# Physically impossible readings
# AH = ambient humidity (%), must be within [0, 100]. CO/NOX concentrations cannot be negative.
impossible_ah = ~df['AH'].between(0, 100)
impossible_co = df['CO'] < 0
impossible_nox = df['NOX'] < 0

print(f"AH readings above 100% : {impossible_ah.sum()} rows ({impossible_ah.mean()*100:.2f}%)")
print(f"Negative CO readings   : {impossible_co.sum()} rows")
print(f"Negative NOX readings  : {impossible_nox.sum()} rows")

AH readings above 100% : 478 rows (1.30%)
Negative CO readings   : 0 rows
Negative NOX readings  : 0 rows


**Decision:** ambient humidity above 100% is physically impossible (relative humidity is capped at 100% by definition), and it affects a small but non-trivial slice of the data (~1.3%). Treat this as a sensor calibration artefact rather than dropping the rows outright (dropping ~1.3% of a time-ordered dataset would create small gaps), and **cap `AH` at 100** before modelling. CO and NOX have no negative readings, so no action is needed there.

In [40]:
df['AH'] = df['AH'].clip(upper=100)
print('AH range after capping:', df['AH'].min(), '-', df['AH'].max())

AH range after capping: 24.085 - 100.0


In [41]:
# Statistical outliers via IQR — reported only, not removed (see reasoning above)
num_cols = [c for c in df.columns if c != 'year']
outlier_report = {}
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[c] < lo) | (df[c] > hi)).sum())
    outlier_report[c] = n_out

pd.Series(outlier_report, name='IQR outlier count').sort_values(ascending=False)

TAT     4955
CO      2655
NOX      936
AP       612
AFDP     557
TIT      315
AH       132
TEY       33
CDP       10
GTEP       7
AT         1
Name: IQR outlier count, dtype: int64

Note the two columns with the most IQR-flagged points: `TAT` and `CO`. Both are expected to have a skewed / multi-modal distribution driven by turbine load (low-load operation produces distinctly different exhaust temperatures and CO levels than high-load operation) rather than measurement error - this will be visualised properly in the EDA notebook. Keep these rows.

## Physics-motivated feature engineering (candidates)

Two engineered features, each with a physical rationale rather than blind combinatorics:

- **`TDROP` = `TIT` − `TAT`** — the temperature drop across the turbine, a direct proxy for the work extracted from the gas path. NOx formation is driven by combustion conditions that the inlet and after temperatures only describe jointly; their difference captures the energy-conversion intensity in a single variable.
- **`PRATIO` = `CDP` / `AP`** — the overall pressure ratio (compressor discharge normalised by ambient pressure), a dimensionless operating-point descriptor. Using the ratio rather than raw `CDP` removes day-to-day weather variation in ambient pressure from the load signal.

Both are created here so they flow through the saved train/val/test splits. Whether they are actually used is decided in Notebook 3: models are compared with and without them on the validation set, and the outcome is recorded in `artifacts/features.json`.

In [42]:
df['TDROP'] = df['TIT'] - df['TAT']
df['PRATIO'] = df['CDP'] / df['AP']

df[['TDROP', 'PRATIO']].describe().T

,count,mean,std,min,25%,50%,75%,max
TDROP,36733.0,535.269567,21.112597,483.260000,522.060000,536.020000,550.790000,586.91000
PRATIO,36733.0,0.011905,0.001068,0.009724,0.011268,0.011824,0.012701,0.01482


## Chronological train / validation / test split

No shuffling: the data is a time-ordered sequence of hourly readings, so a random split would leak information from the future into training. Split by year instead:
- **Train:** 2011-2013
- **Validation:** 2014
- **Test:** 2015 (held out, only touched in the evaluation notebook)

In [43]:
train = df[df['year'].isin([2011, 2012, 2013])].reset_index(drop=True)
val = df[df['year'] == 2014].reset_index(drop=True)
test = df[df['year'] == 2015].reset_index(drop=True)

print('Train:', train.shape)
print('Val:  ', val.shape)
print('Test: ', test.shape)

Train: (22191, 14)
Val:   (7158, 14)
Test:  (7384, 14)


In [ ]:
train.to_csv(ARTIFACT_DIR / 'train.csv', index=False)
val.to_csv(ARTIFACT_DIR / 'val.csv', index=False)
test.to_csv(ARTIFACT_DIR / 'test.csv', index=False)
print('Saved train/val/test to', ARTIFACT_DIR.resolve())